# Task 4: Google Play Store App Category Cumulative Installs Analysis

This notebook demonstrates the complete data engineering and visualization pipeline to construct a stacked area chart showing the cumulative number of installs over time for filtered app categories.

## Objectives & Requirements
1. **Filter Criteria**:
   - Rating $\ge 4.2$
   - App names containing no numbers
   - App categories starting with 'T' or 'P'
   - Reviews $> 1,000$
   - App sizes between 20 MB and 80 MB
2. **Legend Translations**:
   - "Travel & Local" $\rightarrow$ French ("Voyage et guides locaux")
   - "Productivity" $\rightarrow$ Spanish ("Productividad")
   - "Photography" $\rightarrow$ Japanese ("写真")
3. **High Growth Highlighting**:
   - Highlight months with month-over-month (MoM) growth $> 25\%$ for any category by increasing color intensity.

### Step 1: Import Packages & Configure Font Settings
We set up CJK font support in Matplotlib to render the Japanese characters in the legend correctly.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

# Set font for CJK support (Japanese characters like '写真')
plt.rcParams['font.sans-serif'] = ['MS Gothic', 'Yu Gothic', 'Segoe UI Historic', 'Segoe UI', 'DejaVu Sans', 'Arial']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

### Step 2: Load and Clean Dataset
We load the csv dataset, drop corrupt rows (such as index 10472 where Category is '1.9'), and convert numeric columns.

In [ ]:
df = pd.read_csv('Dataset/Play Store Data.csv')
print(f"Raw dataset shape: {df.shape}")

# Drop corrupt rows (e.g. category '1.9')
df = df[df['Category'] != '1.9']
print(f"Dataset shape after removing corrupt category: {df.shape}")

### Step 3: Implement Custom Parsing Functions
We parse reviews, convert size representations (M for Megabytes, k for Kilobytes) to standard MB units, and strip installs characters (`+` and `,`).

In [ ]:
# Parse Reviews to numeric
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

# Parse App Sizes to MB
def parse_size(size_str):
    if not isinstance(size_str, str):
        return None
    size_str = size_str.strip().upper()
    if size_str == 'VARIES WITH DEVICE':
        return None
    if size_str.endswith('M'):
        return float(size_str[:-1])
    if size_str.endswith('K'):
        return float(size_str[:-1]) / 1024.0
    return None

df['Size_MB'] = df['Size'].apply(parse_size)

# Parse Installs to float
def parse_installs(installs_str):
    if not isinstance(installs_str, str):
        return None
    installs_str = installs_str.replace('+', '').replace(',', '').strip()
    return float(installs_str)

df['Installs_numeric'] = df['Installs'].apply(parse_installs)

# Parse Last Updated to datetime
df['Last Updated Date'] = pd.to_datetime(df['Last Updated'], errors='coerce')

### Step 4: Apply Strict Filters
We filter applications by rating, name checks, category characters, reviews count, and sizes.

In [ ]:
f1 = df['Rating'] >= 4.2
f2 = df['App'].apply(lambda x: not any(c.isdigit() for c in str(x)))
f3 = df['Category'].str.startswith(('T', 'P'))
f4 = df['Reviews'] > 1000
f5 = (df['Size_MB'] >= 20.0) & (df['Size_MB'] <= 80.0)

filtered_df = df[f1 & f2 & f3 & f4 & f5].copy()
filtered_df = filtered_df.dropna(subset=['Last Updated Date'])
filtered_df['YearMonth'] = filtered_df['Last Updated Date'].dt.to_period('M')

print(f"Filtered apps matching all conditions: {len(filtered_df)}")
print("\nApp count per category:")
print(filtered_df['Category'].value_counts())

### Step 5: Construct Timeline and Calculate Cumulative Installs & MoM Growth
Since data can be sparse for certain months, we construct a continuous monthly grid from the first recorded date to the last, reindex our grouping, compute the cumulative installs, and then calculate month-over-month growth.

In [ ]:
min_date = filtered_df['Last Updated Date'].min()
max_date = filtered_df['Last Updated Date'].max()
all_months = pd.period_range(start=min_date.to_period('M'), end=max_date.to_period('M'), freq='M')
categories = sorted(filtered_df['Category'].unique())

# Create MultiIndex grid
idx = pd.MultiIndex.from_product([categories, all_months], names=['Category', 'YearMonth'])
grid_df = pd.DataFrame(index=idx).reset_index()

# Aggregate monthly installs
monthly_installs = filtered_df.groupby(['Category', 'YearMonth'])['Installs_numeric'].sum().reset_index()
merged = pd.merge(grid_df, monthly_installs, on=['Category', 'YearMonth'], how='left').fillna(0)

# Cumulative sum over time
merged['Cumulative_Installs'] = merged.groupby('Category')['Installs_numeric'].cumsum()

# MoM Growth Rate
merged['Prev_Cumulative'] = merged.groupby('Category')['Cumulative_Installs'].shift(1)
merged['MoM_Increase_Pct'] = (merged['Cumulative_Installs'] - merged['Prev_Cumulative']) / merged['Prev_Cumulative']

# Treat initial entries as growth (>25% / infinity)
merged.loc[merged['Prev_Cumulative'].isna() & (merged['Cumulative_Installs'] > 0), 'MoM_Increase_Pct'] = float('inf')
merged.loc[(merged['Prev_Cumulative'] == 0) & (merged['Cumulative_Installs'] > 0), 'MoM_Increase_Pct'] = float('inf')

print("Preview of processed cumulative installs & MoM increase:")
print(merged.tail(10))

### Step 6: Map Translations & Prepare Colors
We define the required translation mappings for the chart legend, translate the category names, and assign a unique base color for each category.

In [ ]:
category_translations = {
    'TRAVEL_AND_LOCAL': 'Voyage et guides locaux',  # French
    'PRODUCTIVITY': 'Productividad',               # Spanish
    'PHOTOGRAPHY': '写真',                         # Japanese
    'TOOLS': 'Tools',                              # Unchanged
    'PERSONALIZATION': 'Personalization',          # Unchanged
    'PARENTING': 'Parenting'                       # Unchanged
}

merged['Category_Translated'] = merged['Category'].map(category_translations)

# Pivot the tables for plotting
pivot_df = merged.pivot(index='YearMonth', columns='Category', values='Cumulative_Installs')
pivot_df.index = pivot_df.index.to_timestamp()

mom_pivot = merged.pivot(index='YearMonth', columns='Category', values='MoM_Increase_Pct')
mom_pivot.index = mom_pivot.index.to_timestamp()

categories_translated_sorted = [category_translations[cat] for cat in categories]

colors = {
    'Voyage et guides locaux': '#1f77b4', # blue
    'Tools': '#9467bd',                 # purple
    'Productividad': '#2ca02c',          # green
    '写真': '#ff7f0e',                   # orange
    'Personalization': '#e377c2',        # pink
    'Parenting': '#bcbd22'               # olive/yellow
}

### Step 7: Plot the Stacked Area Chart with Month-Specific Growth Highlights
We use Plotly to create an interactive stacked area chart with growth highlights.

In [ ]:
import plotly.graph_objects as go
import os

# Setup Plotly fig
fig = go.Figure()

# Accumulate heights manually to build stacked traces
y_cum = np.zeros(len(pivot_df.index))
y_stacked_cum = np.vstack([np.zeros(len(pivot_df.index))] + [pivot_df[cat].values for cat in categories]).cumsum(axis=0)
for cat in categories:
    trans_name = category_translations[cat]
    val = pivot_df[cat].values
    y_next = y_cum + val
    
    # Format x-values as ISO string dates to avoid serialization issues
    fig.add_trace(go.Scatter(
        x=[t.strftime("%Y-%m-%d") for t in pivot_df.index],
        y=y_next,
        fill='tonexty' if fig.data else 'tozeroy',
        mode='none',
        name=trans_name,
        fillcolor=colors[trans_name]
    ))
    y_cum = y_next

# Add vertical shapes for high growth months in Plotly
for month_timestamp in pivot_df.index:
    is_high_growth_month = False
    for cat in categories:
        if mom_pivot.loc[month_timestamp, cat] > 0.25:
            is_high_growth_month = True
            break
    
    if is_high_growth_month:
        x0 = month_timestamp - pd.Timedelta(days=15)
        x1 = month_timestamp + pd.Timedelta(days=15)
        
        fig.add_vrect(
            x0=x0.strftime("%Y-%m-%d"), x1=x1.strftime("%Y-%m-%d"),
            fillcolor="#a855f7", opacity=0.12,
            layer="below", line_width=0
        )

# Add growth annotations (🚀 +XX%) on Plotly for major spikes
for c_idx, cat in enumerate(categories):
    for i in range(1, len(pivot_df.index)):
        month_timestamp = pivot_df.index[i]
        growth = mom_pivot.loc[month_timestamp, cat]
        prev_val = pivot_df.loc[pivot_df.index[i-1], cat]
        
        if 0.50 < growth < float('inf') and prev_val > 1000000:
            growth_pct = int(round(growth * 100))
            y_val = y_stacked_cum[c_idx+1, i]
            
            fig.add_annotation(
                x=month_timestamp.strftime("%Y-%m-%d"),
                y=y_val,
                text=f"🚀 +{growth_pct}%",
                showarrow=True,
                arrowhead=2,
                arrowcolor="#FFD700",
                arrowsize=1,
                arrowwidth=1.5,
                ax=20,
                ay=-25 if cat != 'TRAVEL_AND_LOCAL' else 25,
                font=dict(size=9, color="#FFD700", family="Inter, sans-serif"),
                bgcolor="#1f2235",
                bordercolor="#FFD700",
                borderwidth=1,
                borderpad=4,
                opacity=0.9
            )

fig.update_layout(
    title="Cumulative Installs over Time (Interactive Plotly Stacked Area)",
    xaxis_title="Time (Month/Year)",
    yaxis_title="Cumulative Installs",
    template="plotly_dark",
    paper_bgcolor="#0e1117",
    plot_bgcolor="#0e1117",
    legend=dict(
        title="App Categories",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(30, 34, 53, 0.7)"
    )
)

os.makedirs('Screenshots', exist_ok=True)
fig.write_image('Screenshots/Graph1.png', width=1400, height=800, scale=2)
print("Plotly chart saved as static image Screenshots/Graph1.png successfully!")
fig.show()